# Optuna F5-Alpha Study — Colab H100

Optimizes 3 parameters against the 20-seq SUBSET:
- **f5_alpha** `[0.0, 1.0]` — F5 closed-loop blending factor  
- **q_scale** `[5, 200]` — IMM process noise scale  
- **r_pos_scale** `[5, 200]` — IMM position measurement noise

Baseline: `FS_imm = 0.7139` (i12 + T1 + F5 all-frames)  
Pruning: car8 and Paragliding3 canary check before full 20-seq eval.

**Before running:** upload your repo to Google Drive and set `REPO_PATH` below.

In [ ]:
# ── 0. Config — adjust these paths ────────────────────────────────────────
import os

REPO_PATH   = "/content/drive/MyDrive/tracker"   # repo root on your Drive
N_TRIALS    = 60
STUDY_NAME  = "f5alpha_qscale_rpos_h100"
BUILD_DIR   = "build_colab"                       # CMake output dir

# Study DB (SQLite) — resumable if Colab session drops
DB_PATH     = os.path.join(REPO_PATH, "cache", "optuna_studies", f"{STUDY_NAME}.db")
CKPT_DIR    = os.path.join(REPO_PATH, "cache", "optuna_studies")  # best YAML saved here
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"DB   : {DB_PATH}")
print(f"CKPT : {CKPT_DIR}")


In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. System deps (C++ build tools + OpenCV) ─────────────────────────────
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

print("Installing system packages...")
run("apt-get install -y cmake build-essential libopencv-dev python3-dev > /dev/null 2>&1")
print("Done.")

In [ ]:
# ── 3. Python deps ────────────────────────────────────────────────────────
!pip install -q optuna onnxruntime-gpu
# Verify CUDA
import torch
print(f"CUDA: {torch.cuda.is_available()}  |  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── 4. Build C++ tracker_cpp extension ────────────────────────────────────
# TRT engines are device-specific — we use SGLATrackWrapper (PyTorch) here.
# tracker_cpp (KalmanFilter, IMMFilter, etc.) still needs to be compiled.

import os
os.chdir(REPO_PATH)

BUILD_DIR = "build_colab"

print("Configuring CMake...")
run(f"cmake -B {BUILD_DIR} -DCMAKE_BUILD_TYPE=Release > /dev/null 2>&1")
print("Building (this takes ~5 min on first run)...")
run(f"cmake --build {BUILD_DIR} -j$(nproc) --target tracker_cpp 2>&1 | tail -5")
print("Build complete.")

In [ ]:
# ── 5. Verify imports ─────────────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.join(REPO_PATH, "python"))
sys.path.insert(0, os.path.join(REPO_PATH, BUILD_DIR))

import tracker_cpp
print(f"tracker_cpp loaded: {tracker_cpp}")

from tracker.sglatrack_wrapper import SGLATrackWrapper
t = SGLATrackWrapper()
print(f"SGLATrackWrapper OK")

In [ ]:
# ── 6. Verify data access ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.join(REPO_PATH, "python"))
sys.path.insert(0, os.path.join(REPO_PATH, BUILD_DIR))

from tracker.data_utils import load_manifest
manifest = load_manifest(data_root=os.path.join(REPO_PATH, "data", "contest_release"))
n_seqs = sum(len(v) for v in manifest.values())
assert n_seqs > 200, f"Expected >200 seqs, got {n_seqs}"
print(f"Manifest OK: {n_seqs} sequences")


In [ ]:
# ── 7. Run Optuna study ────────────────────────────────────────────────────
# Resumes automatically if DB already exists (safe to re-run after disconnect)
import subprocess, sys

cmd = [
    sys.executable,
    os.path.join(REPO_PATH, "scripts", "optuna_f5alpha.py"),
    "--use-sgla",
    "--n-trials",    str(N_TRIALS),
    "--study-name",  STUDY_NAME,
    "--study-db",    DB_PATH,
    "--checkpoint-dir", CKPT_DIR,
    "--gmc",
    "--adaptive-r",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_PATH)
print(f"\nExit code: {result.returncode}")


In [ ]:
# ── 8. Visualize results ──────────────────────────────────────────────────
import optuna

study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=f"sqlite:///{DB_PATH}",
)

completed_trials = sorted(
    [t for t in study.trials if t.value is not None],
    key=lambda t: t.value, reverse=True
)
n_pruned = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
print(f"Completed: {len(completed_trials)}  |  Pruned: {n_pruned}")

if not completed_trials:
    print("No completed trials yet — all pruned by canary gate.")
else:
    bt = completed_trials[0]
    print(f"Best FS_imm : {bt.value:.4f}  (Δ vs baseline 0.7139: {bt.value - 0.7139:+.4f})")
    print(f"  f5_alpha    = {bt.params['f5_alpha']:.4f}")
    print(f"  q_scale     = {bt.params['q_scale']:.2f}")
    print(f"  r_pos_scale = {bt.params['r_pos_scale']:.2f}")
    print("\nTop-5:")
    for t in completed_trials[:5]:
        print(f"  #{t.number:<4} FS={t.value:.4f}  f5α={t.params['f5_alpha']:.3f}"
              f"  q={t.params['q_scale']:.1f}  rp={t.params['r_pos_scale']:.1f}")
    try:
        importances = optuna.importance.get_param_importances(study)
        print("\nParam importances:", importances)
    except Exception as e:
        print(f"Importance calc skipped: {e}")


In [ ]:
# ── 9. Generate n1_f5alpha_optuna.yaml with best params ──────────────────
import yaml, copy, shutil

if not completed_trials:
    print("No completed trials — cannot generate config.")
else:
    BASE_CFG = os.path.join(REPO_PATH, "configs", "i12_rescue_area_gate.yaml")
    OUT_CFG  = os.path.join(REPO_PATH, "configs", "n1_f5alpha_optuna.yaml")

    with open(BASE_CFG) as f:
        cfg = yaml.safe_load(f)

    bt = completed_trials[0]  # already sorted by value desc
    cfg["imm"]["q_scale"] = round(bt.params["q_scale"], 3)
    cfg["imm"]["measurement_noise"]["r_pos_scale"] = round(bt.params["r_pos_scale"], 3)
    cfg.setdefault("ai", {})["f5_alpha"] = round(bt.params["f5_alpha"], 4)

    with open(OUT_CFG, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

    print(f"Config written: {OUT_CFG}")
    print(f"  FS_imm={bt.value:.4f}  f5_alpha={bt.params['f5_alpha']:.4f}"
          f"  q={bt.params['q_scale']:.2f}  rp={bt.params['r_pos_scale']:.2f}")
    print("\nNext step (run locally):")
    print("  python3 scripts/ab_test.py --imm-config configs/n1_f5alpha_optuna.yaml"
          " --gmc --adaptive-r --mode ai_lead")
    # Also copy to CKPT_DIR as named backup
    ckpt_copy = os.path.join(CKPT_DIR, f"{STUDY_NAME}_final_best.yaml")
    shutil.copy2(OUT_CFG, ckpt_copy)
    print(f"Backup: {ckpt_copy}")
